In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import random

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Dataset: rotation pretext (keep MNIST label too)
# -----------------------------
class RotationDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.rotation_angles = [0, 90, 180, 270]
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, digit_label = self.base_dataset[idx]
        angle_idx = random.randint(0, 3)
        rotated_img = transforms.functional.rotate(img, self.rotation_angles[angle_idx])
        x = self.to_tensor(rotated_img)
        return x, angle_idx, digit_label

# -----------------------------
# Model: encoder + rotation head
# -----------------------------
class Encoder(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, emb_dim),
            nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)

class RotationHead(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        self.fc = nn.Linear(emb_dim, 4)
    def forward(self, h):
        return self.fc(h)

class RotationModel(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        self.encoder = Encoder(emb_dim)
        self.head = RotationHead(emb_dim)
    def forward(self, x):
        h = self.encoder(x)
        logits = self.head(h)
        return logits, h

# -----------------------------
# 1) Pretext training (rotation)
# -----------------------------
base_dataset = datasets.MNIST(root="./data", train=True, download=True)
subset = Subset(base_dataset, list(range(2000)))   # più grande di 200 per imparare qualcosa
rot_ds = RotationDataset(subset)
rot_loader = DataLoader(rot_ds, batch_size=128, shuffle=True)

model = RotationModel(emb_dim=256).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
ce = nn.CrossEntropyLoss()

model.train()
for epoch in range(5):
    total = 0.0
    for x, angle_y, digit_y in rot_loader:
        x, angle_y = x.to(device), angle_y.to(device)
        logits, h = model(x)
        loss = ce(logits, angle_y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"[Pretext] epoch {epoch+1} loss {total/len(rot_loader):.4f}")

# -----------------------------
# 2) Freeze encoder
# -----------------------------
encoder = model.encoder
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

# -----------------------------
# 3) Supervised digit classifier on frozen embeddings
# -----------------------------
class DigitClassifier(nn.Module):
    def __init__(self, emb_dim=256, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(emb_dim, num_classes)
    def forward(self, h):
        return self.fc(h)

clf = DigitClassifier(emb_dim=256).to(device)
opt_clf = torch.optim.Adam(clf.parameters(), lr=1e-3)
ce_digit = nn.CrossEntropyLoss()

# Use NON-rotated MNIST for digit labels (clean supervised task)
sup_transform = transforms.ToTensor()
train_sup = datasets.MNIST(root="./data", train=True, download=True, transform=sup_transform)
train_sup = Subset(train_sup, list(range(5000)))
sup_loader = DataLoader(train_sup, batch_size=128, shuffle=True)

clf.train()
for epoch in range(5):
    total = 0.0
    correct = 0
    n = 0
    for x, y in sup_loader:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            h = encoder(x)  # frozen embedding

        logits = clf(h)
        loss = ce_digit(logits, y)

        opt_clf.zero_grad()
        loss.backward()
        opt_clf.step()

        total += loss.item()
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        n += y.size(0)

    print(f"[Supervised head] epoch {epoch+1} loss {total/len(sup_loader):.4f} acc {correct/n:.3f}")

# -----------------------------
# 4) t-SNE colored by MNIST class (using embeddings)
# -----------------------------
# small subset for visualization
viz_ds = Subset(train_sup, list(range(1000)))
viz_loader = DataLoader(viz_ds, batch_size=256, shuffle=False)

features, labels = [], []
encoder.eval()
with torch.no_grad():
    for x, y in viz_loader:
        x = x.to(device)
        h = encoder(x).cpu()
        features.append(h)
        labels.append(y)

features = torch.cat(features).numpy()
labels = torch.cat(labels).numpy()

tsne = TSNE(
    n_components=2,
    init="random",
    learning_rate="auto"
)
emb2d = tsne.fit_transform(features)

plt.figure(figsize=(8,6))
for c in range(10):
    idx = labels == c
    plt.scatter(emb2d[idx,0], emb2d[idx,1], s=12, alpha=0.6, label=str(c))
plt.legend(title="MNIST class", ncol=2)
plt.title("t-SNE of embeddings learned via rotation pretext (colored by digit)")
plt.show()